**LSTM FINANCIAL TIME SERIES MODEL ON NIFTY50 AND S&P500**

In [243]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

**DOWNLOADING DATA**

In [244]:
nifty = yf.download('^NSEI', start='2010-01-01', end='2024-01-01')
sp500 = yf.download('^GSPC', start='2010-01-01', end='2024-01-01')

/tmp/ipykernel_2275/1034130995.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  nifty = yf.download('^NSEI', start='2010-01-01', end='2024-01-01')
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_2275/1034130995.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500 = yf.download('^GSPC', start='2010-01-01', end='2024-01-01')
[*********************100%***********************]  1 of 1 completed


In [245]:
print(nifty.head())
print(sp500.head())

Price             Close         High          Low         Open Volume
Ticker            ^NSEI        ^NSEI        ^NSEI        ^NSEI  ^NSEI
Date                                                                 
2010-01-04  5232.200195  5238.450195  5167.100098  5200.899902      0
2010-01-05  5277.899902  5288.350098  5242.399902  5277.149902      0
2010-01-06  5281.799805  5310.850098  5260.049805  5278.149902      0
2010-01-07  5263.100098  5302.549805  5244.750000  5281.799805      0
2010-01-08  5244.750000  5276.750000  5234.700195  5264.250000      0
Price             Close         High          Low         Open      Volume
Ticker            ^GSPC        ^GSPC        ^GSPC        ^GSPC       ^GSPC
Date                                                                      
2010-01-04  1132.989990  1133.869995  1116.560059  1116.560059  3991400000
2010-01-05  1136.520020  1136.630005  1129.660034  1132.660034  2491020000
2010-01-06  1137.140015  1139.189941  1133.949951  1135.709961  4

In [246]:
print(nifty.shape)
print(sp500.shape)

(3434, 5)
(3522, 5)


In [247]:
nifty.isnull().sum().sum()
sp500.isnull().sum().sum()

np.int64(0)

**FEATURE ENGINEERING**

In [248]:
nifty['returns'] = nifty['Close']['^NSEI'].pct_change(fill_method=None)

In [249]:
volume_nifty = nifty['Volume']['^NSEI']

In [250]:
nifty['Volume Ratio'] = (volume_nifty/volume_nifty.rolling(10).mean()).shift(1)

In [251]:
nifty['Rolling Returns'] = nifty['returns'].rolling(20).mean().shift(1)

In [252]:
delta = nifty['Close']['^NSEI'].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
RSI = 100 - (100 / (1 + gain/loss))
nifty['RSI'] = RSI.shift(1)

In [253]:
sp500['SP500 returns'] = sp500['Close']['^GSPC'].pct_change(fill_method=None)

In [254]:
sp500_return_dataframe = pd.DataFrame({
    'Date': sp500.index,
    'SP500 returns': sp500['SP500 returns']
})
sp500_return_dataframe.set_index('Date', inplace=True)

In [255]:
nifty_clean = pd.DataFrame({
    "returns": nifty['returns'],
    "Volume Ratio": nifty['Volume Ratio'],
    "Rolling Returns": nifty['Rolling Returns'],
    "RSI": nifty['RSI']
})

In [256]:
nifty_clean = nifty_clean.join(sp500_return_dataframe,how="inner")

In [257]:
nifty_clean['target'] = (nifty_clean['returns'] > 0).astype(int).shift(-1)
nifty_clean.dropna(inplace=True)
nifty_clean

,returns,Volume Ratio,Rolling Returns,RSI,SP500 returns,target
Date,,,,,,
2013-01-22,-0.005557,10.000000,0.001405,69.999945,0.004428,1.0
2013-01-23,0.000959,4.963447,0.001708,57.107931,0.001507,0.0
2013-01-24,-0.005773,3.451751,0.001687,55.922763,0.000007,1.0
2013-01-25,0.009187,3.181584,0.000973,50.393628,0.005445,1.0
2013-01-28,0.000025,2.022749,0.001733,59.935496,-0.001850,0.0
...,...,...,...,...,...,...
2023-12-21,0.004960,1.216203,0.003371,76.031608,0.010301,1.0
2023-12-22,0.004439,0.926668,0.003547,75.659666,0.001660,1.0
2023-12-26,0.004307,0.953162,0.003794,70.719263,0.004232,1.0


In [258]:
nifty_clean.shape

(2614, 6)

In [259]:
nifty_clean.columns.tolist()

['returns',
 'Volume Ratio',
 'Rolling Returns',
 'RSI',
 'SP500 returns',
 'target']

**PREPARATION OF DATA FOR THE LSTM MODEL**

In [260]:
features = nifty_clean.drop('target',axis=1)
target = nifty_clean['target']

In [261]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

In [262]:
target = target.to_numpy()

In [263]:
lookback = 20
x,y = [] , []

for i in range(len(features_scaled) - lookback):
  x.append(features_scaled[i:i+lookback])
  y.append(target[i+lookback])

x = np.array(x)
y = np.array(y)

In [264]:
print(x.shape)
print(y.shape)

(2594, 20, 5)
(2594,)


In [265]:
x_train = x[:int(0.8*len(x))]
y_train = y[:int(0.8*len(y))]
x_test = x[int(0.8*len(x)):]
y_test = y[int(0.8*len(y)):]

In [266]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(2075, 20, 5)
(2075,)
(519, 20, 5)
(519,)


**INITIATING THE MODEL**

In [267]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x):
        output , _ = self.lstm(x)
        out = output[:,-1,:]
        out = self.dropout(out)
        out = self.fc(out)
        return self.sigmoid(out)


In [268]:
model = LSTMModel(5,64)
print(model)

LSTMModel(
  (lstm): LSTM(5, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
  (dropout): Dropout(p=0.3, inplace=False)
)


In [269]:
x_train_t = torch.tensor(x_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_train_t = y_train_t.reshape(-1,1)

x_test_t = torch.tensor(x_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)
y_test_t = y_test_t.reshape(-1,1)


# Loss and optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [270]:
epochs = 100
batch_size = 32

for epoch in range(epochs):
    model.train()
    for i in range(0, len(x_train_t), batch_size):
        x_batch = x_train_t[i:i+batch_size]
        y_batch = y_train_t[i:i+batch_size]
        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    if epoch % 10 == 0:
      print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

Epoch 0, Loss: 0.6896
Epoch 10, Loss: 0.6858
Epoch 20, Loss: 0.6437
Epoch 30, Loss: 0.4984
Epoch 40, Loss: 0.4795
Epoch 50, Loss: 0.4246
Epoch 60, Loss: 0.4058
Epoch 70, Loss: 0.3352
Epoch 80, Loss: 0.3314
Epoch 90, Loss: 0.1865


In [271]:
model.eval()
with torch.no_grad():
    y_pred_train = model(x_train_t)
    y_pred_train_class = (y_pred_train > 0.5).float()
    train_accuracy = (y_pred_train_class == y_train_t).float().mean()
    print(f'Train Accuracy: {train_accuracy.item():.4f}')

Train Accuracy: 0.9239


In [272]:
model.eval()
with torch.no_grad():
    y_pred = model(x_test_t)
    y_pred_class = (y_pred > 0.5).float()
    accuracy = (y_pred_class == y_test_t).float().mean()
    print(f'Test Accuracy: {accuracy.item():.4f}')

Test Accuracy: 0.4933
